# Track A 05: Full Frozen Llama Quality and Matched-Wrapper Control

This is the first full run of the accepted V4 hard-retrieval prompt set. It is self-contained and writes only to `CSE407_llama_track_a_full_v1/`.

| Label | Path |
|---|---|
| C0 | FP16 stock attention |
| C1-S | AWQ stock attention, parity control only |
| C1-W | AWQ matched wrapper, no cache compression |
| C2 | AWQ + custom pyramidal eviction |
| C3-Ref | AWQ + `turboquant==0.2.0` numerical reference path |
| C4-Ref | AWQ + sequential eviction then TQ-Ref |

TQ-Ref is not packed, fused, or physically 4-bit. The quality matrix is exactly 300 rows: five final variants by 60 frozen scenarios. Pilot rows and historical easy-NIAH rows are never loaded.

## Run Order and Safety Boundary

Run cells in numeric order. Frozen hashes are regenerated and asserted before any model is loaded. The AWQ parity and cache-path validation gate must pass before measurements. FP16 and AWQ remain explicit model-resident phases.

This source notebook has no fabricated outputs and is intentionally unexecuted. Never change the generator, expected hashes, schema, fingerprint inputs, or output directory after collecting any row.

## 1. Install Pinned Dependencies

Run once, choose **Runtime > Restart session**, and continue with cell 2.

In [ ]:
%pip install -q "autoawq==0.2.9" "gptqmodel==7.1.0" "turboquant==0.2.0"
%pip install -q "transformers==5.12.1" accelerate nvidia-ml-py pandas scipy

## 2. Runtime, Imports, and Immutable Protocol Constants

Select a T4 GPU runtime. The schema and expected frozen hashes are predeclared. The experiment fingerprint is derived only from fixed protocol values and is checked against the registered value.

In [ ]:
import gc
import hashlib
import importlib.metadata
import json
import math
import os
import random
import re
import threading
import time
from collections.abc import Mapping
from pathlib import Path

import numpy as np
import pandas as pd
import pynvml
import scipy.stats as stats
import torch
import torch.nn.functional as F
import transformers
import turboquant
from transformers import AutoModelForCausalLM, AutoTokenizer, DynamicCache, set_seed
from transformers.modeling_utils import ALL_ATTENTION_FUNCTIONS
from transformers.models.llama.modeling_llama import (
    LlamaAttention,
    apply_rotary_pos_emb,
    eager_attention_forward,
    repeat_kv,
)
from turboquant import TurboQuantCache, TurboQuantMSE

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

SEED = 407
SCHEMA_VERSION = "track-a-llama-full-v1-2026-08-03-1"
EXPECTED_FINGERPRINT = "106a561c07c62937"
MODEL_FP16 = "unsloth/Llama-3.2-3B-Instruct"
MODEL_FP16_REVISION = "006f5dcd1393c3add266de40994ba96225e9689d"
MODEL_AWQ = "casperhansen/llama-3.2-3b-instruct-awq"
MODEL_AWQ_REVISION = "272b3bde867b606760447deb9a4d2719fbdfd3ae"
EXPECTED_TQ_SOURCE_SHA256 = "7358e2778c8de6974b0c7212ae8e3dbe00552a96b2e2df0aa42878f239a4c72a"
EXPECTED_SEMANTIC_SHA256 = "fffa82607af6e7d2c892d18d5bc790524570833472ad397b7ab2e54eb03005ef"
EXPECTED_PROMPT_SET_SHA256 = "4e1ced4fd4829629929a346879cc00c45476f38fa0936137909b8779242321e8"
CONTEXT_TOKENS = 4096
MAX_NEW_TOKENS = 32
FIXED_DECODE_TOKENS = 64
MEASURED_BLOCKS = 5
DEFAULT_AVERAGE_BUDGET = 1024
DEFAULT_TQ_BITS = 4
NUM_LAYERS = 28
NUM_Q_HEADS = 24
NUM_KV_HEADS = 8
HEAD_DIM = 128
WINDOW = 32
POOL_KERNEL = 5
PYRAMID_BETA = 20
PARITY_ATOL = 5e-3
PARITY_RTOL = 5e-3
QUALITY_VARIANTS = ("C0", "C1-W", "C2", "C3-Ref", "C4-Ref")
AWQ_QUALITY_VARIANTS = ("C1-W", "C2", "C3-Ref", "C4-Ref")
FORBIDDEN_ARTIFACT_NAMES = {
    "CSE407_llama_v5_final",
    "CSE407_llama_track_a_pilot_discarded",
    "CSE407_llama_track_a_pilot_v2_discarded",
    "CSE407_llama_track_a_pilot_v3_discarded",
    "CSE407_llama_track_a_pilot_v4_discarded",
}

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
assert torch.cuda.is_available(), "A CUDA GPU is required"
GPU = torch.cuda.get_device_properties(0)
assert "T4" in GPU.name, f"Expected an NVIDIA T4, found {GPU.name!r}"
assert transformers.__version__ == "5.12.1"
assert importlib.metadata.version("turboquant") == "0.2.0"
assert importlib.metadata.version("gptqmodel") == "7.1.0"

TQ_PACKAGE_DIR = Path(turboquant.__file__).parent
TQ_SOURCE_SHA256 = hashlib.sha256(
    b"".join(path.read_bytes() for path in sorted(TQ_PACKAGE_DIR.glob("*.py")))
).hexdigest()
assert TQ_SOURCE_SHA256 == EXPECTED_TQ_SOURCE_SHA256

pynvml.nvmlInit()
NVML_HANDLE = pynvml.nvmlDeviceGetHandleByIndex(0)
SESSION_ID = time.strftime("%Y%m%dT%H%M%S") + "-" + os.urandom(4).hex()

USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    SAVE_DIR = Path("/content/drive/MyDrive/CSE407_llama_track_a_full_v1")
else:
    SAVE_DIR = Path("/content/CSE407_llama_track_a_full_v1")
assert SAVE_DIR.name == "CSE407_llama_track_a_full_v1"
assert SAVE_DIR.name not in FORBIDDEN_ARTIFACT_NAMES
SAVE_DIR.mkdir(parents=True, exist_ok=True)

QUALITY_ROW_DIR = SAVE_DIR / "quality_rows"
ENERGY_ROW_DIR = SAVE_DIR / "energy_rows"
POWER_TRACE_DIR = SAVE_DIR / "power_traces"
for directory in (QUALITY_ROW_DIR, ENERGY_ROW_DIR, POWER_TRACE_DIR):
    directory.mkdir(exist_ok=True)
QUALITY_CSV = SAVE_DIR / "track_a_full_quality_results.csv"
ENERGY_CSV = SAVE_DIR / "track_a_c1w_energy_control.csv"
SEMANTIC_MANIFEST_JSON = SAVE_DIR / "frozen_semantic_manifest.json"
PROMPT_MANIFEST_JSON = SAVE_DIR / "frozen_llama_prompt_manifest.json"
ENVIRONMENT_JSON = SAVE_DIR / "track_a_full_environment.json"
VALIDATION_JSON = SAVE_DIR / "track_a_full_validation.json"
PARITY_JSON = SAVE_DIR / "track_a_c1s_c1w_parity.json"
ANALYSIS_DIR = SAVE_DIR / "analysis"
ANALYSIS_DIR.mkdir(exist_ok=True)
print(GPU.name, "runtime ready; outputs:", SAVE_DIR)

## 3. Embedded Frozen V4 Generator and Scorer

This is the accepted V4 implementation. Its draft status string and ordering are retained because they are part of the frozen semantic hash. Do not refactor or tune it.

In [ ]:
GENERATOR_VERSION = "track-a-hard-retrieval-v4-controlled-pairs-draft"
MANIFEST_SCHEMA = "cse407-hard-retrieval-manifest-v1"
BASE_SEED = 407_100
DEPTHS = (0.1, 0.5, 0.9)
SCENARIOS_PER_DEPTH = 20
RECORD_COUNT = 32
FINAL_QUERY_WINDOW = 32
MIN_RECORD_GAP_TOKENS = 16

CODE_PATTERN = re.compile(
    r"(?<![A-Z0-9])([A-H]\d{3}-[K-NP-Q]\d{2})(?![A-Z0-9])",
    re.IGNORECASE,
)
PARTIAL_CODE_PATTERN = re.compile(
    r"(?<![A-Z0-9])(?:[A-H]\d{1,2}|[A-H]\d{3}(?:-(?:[K-NP-Q]?\d?)?)?)(?![A-Z0-9])",
    re.IGNORECASE,
)
SYSTEM_INSTRUCTION = (
    "Use only the document. Return exactly one activation code and no explanation."
)
PLACEHOLDER = "{HARD_RETRIEVAL_DOCUMENT}"
KEY_LABELS = (
    "AURORA", "BOREAL", "CEDAR", "DRAGON", "EMBER", "FALCON", "GARNET", "HARBOR",
    "INDIGO", "JASPER", "KESTREL", "LANTERN", "MARINER", "NEBULA", "ORBIT", "PRAIRIE",
    "QUARTZ", "RAVEN", "SUMMIT", "TEMPEST", "UMBRA", "VECTOR", "WILLOW", "XENON",
    "YARROW", "ZEPHYR", "ORBITAL", "MARINA", "CEDRIC", "FALCONE", "GARNET-X", "HARBOR-X",
)
FILLER_TOPICS = (
    "thermal scheduling", "memory allocation", "coastal monitoring",
    "warehouse routing", "battery maintenance", "traffic forecasting",
    "compiler testing", "water treatment", "satellite imaging",
    "clinical logistics", "network telemetry", "acoustic sensing",
    "grid balancing", "crop analysis", "robot calibration", "archive indexing",
)
FILLER_METHODS = (
    "compares recent measurements with a rolling baseline",
    "records each decision before the next review cycle",
    "uses independent checks to detect inconsistent observations",
    "separates routine maintenance from exceptional interventions",
    "prioritizes stable operation over short-lived peak performance",
    "tracks uncertainty alongside every summarized measurement",
    "repeats validation after any material configuration change",
    "keeps raw observations available for later auditing",
)
FILLER_OUTCOMES = (
    "The review found that seasonal variation explained most of the change.",
    "Operators retained the conservative threshold for the next evaluation.",
    "No single measurement was treated as sufficient evidence for deployment.",
    "The team documented both successful trials and inconclusive observations.",
    "A later audit confirmed the ordering but revised the uncertainty estimate.",
    "The report emphasized reproducibility and clearly separated derived values.",
    "Follow-up work focused on failure recovery rather than headline throughput.",
    "The final recommendation required another independent measurement period.",
)


def canonical_hash(value):
    payload = json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=True)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def token_ids_hash(input_ids):
    return hashlib.sha256(",".join(map(str, input_ids)).encode("ascii")).hexdigest()


def _make_unique_codes(rng, count):
    letters = "ABCDEFGH"
    suffixes = "KLMNPQ"
    codes = set()
    while len(codes) < count:
        codes.add(
            f"{rng.choice(letters)}{rng.randrange(100, 1000):03d}-"
            f"{rng.choice(suffixes)}{rng.randrange(10, 100):02d}"
        )
    result = sorted(codes)
    rng.shuffle(result)
    return result


def _make_similar_keys(rng, count):
    candidates = [f"NODE-{label}" for label in KEY_LABELS]
    if count > len(candidates):
        raise ValueError("Controlled-pair pilot supports at most 32 records")
    rng.shuffle(candidates)
    return candidates[:count]


def validate_semantic_scenario(scenario):
    records = scenario["records"]
    if not 32 <= len(records) <= 64:
        raise ValueError("Each scenario must contain 32 to 64 records")
    keys = [record["key"] for record in records]
    codes = [record["code"] for record in records]
    if len(set(keys)) != len(keys) or len(set(codes)) != len(codes):
        raise ValueError("Record keys and codes must be unique")
    if any(CODE_PATTERN.fullmatch(code) is None for code in codes):
        raise ValueError("Every record code must match the declared answer format")
    targets = [record for record in records if record["is_target"]]
    if len(targets) != 1:
        raise ValueError("Each scenario must have exactly one target record")
    target = targets[0]
    if target["key"] != scenario["target_key"] or target["code"] != scenario["target_code"]:
        raise ValueError("Target metadata does not match the target record")
    if scenario["depth"] not in DEPTHS:
        raise ValueError(f"Unsupported target depth: {scenario['depth']}")


def build_semantic_manifest(
    depths=DEPTHS,
    scenarios_per_depth=SCENARIOS_PER_DEPTH,
    record_count=RECORD_COUNT,
    base_seed=BASE_SEED,
):
    if not 32 <= record_count <= 64:
        raise ValueError("record_count must be between 32 and 64")
    scenarios = []
    for depth_index, depth in enumerate(depths):
        if depth not in DEPTHS:
            raise ValueError(f"Unsupported target depth: {depth}")
        for index in range(scenarios_per_depth):
            scenario_seed = base_seed + depth_index * 100_000 + index
            rng = random.Random(scenario_seed)
            keys = _make_similar_keys(rng, record_count)
            codes = _make_unique_codes(rng, record_count)
            target_index = rng.randrange(record_count)
            records = [
                {"key": key, "code": code, "is_target": position == target_index}
                for position, (key, code) in enumerate(zip(keys, codes))
            ]
            rng.shuffle(records)
            target = next(record for record in records if record["is_target"])
            scenario = {
                "scenario_id": f"d{int(depth * 100):02d}-s{index:02d}",
                "seed": scenario_seed,
                "depth": depth,
                "target_key": target["key"],
                "target_code": target["code"],
                "records": records,
            }
            validate_semantic_scenario(scenario)
            scenarios.append(scenario)
    manifest = {
        "manifest_schema": MANIFEST_SCHEMA,
        "generator_version": GENERATOR_VERSION,
        "status": "DRAFT_UNTIL_C1W_C2_PILOT_PASSES",
        "base_seed": base_seed,
        "depths": list(depths),
        "scenarios_per_depth": scenarios_per_depth,
        "record_count": record_count,
        "scenarios": scenarios,
    }
    manifest["semantic_sha256"] = canonical_hash(manifest)
    return manifest


def _make_filler_text(rng, minimum_paragraphs=512):
    paragraphs = []
    topic_order = list(FILLER_TOPICS)
    method_order = list(FILLER_METHODS)
    outcome_order = list(FILLER_OUTCOMES)
    for index in range(minimum_paragraphs):
        if index % len(topic_order) == 0:
            rng.shuffle(topic_order)
        if index % len(method_order) == 0:
            rng.shuffle(method_order)
        if index % len(outcome_order) == 0:
            rng.shuffle(outcome_order)
        topic = topic_order[index % len(topic_order)]
        method = method_order[index % len(method_order)]
        outcome = outcome_order[index % len(outcome_order)]
        paragraphs.append(
            f"Memo {index + 1} discusses {topic}. The working group {method}. "
            f"Sample batch {2000 + index} was reviewed by {2 + index % 7} analysts. "
            f"{outcome}\n"
        )
    text = "\n".join(paragraphs)
    if CODE_PATTERN.search(text):
        raise AssertionError("Filler text accidentally contains a code-shaped value")
    return text


def _record_text(record):
    return (
        f"\n[Access record] System {record['key']} uses activation code "
        f"{record['code']}.\n"
    )


def _encode(tokenizer, text):
    return list(tokenizer.encode(text, add_special_tokens=False))


def _allocate_gaps(total, count, rng):
    minimum = MIN_RECORD_GAP_TOKENS * count
    if total < minimum:
        raise ValueError(
            f"Region has {total} filler tokens but needs at least {minimum} "
            "to keep records separated"
        )
    remaining = total - minimum
    gaps = [MIN_RECORD_GAP_TOKENS] * count
    order = list(range(count))
    rng.shuffle(order)
    for index in range(remaining):
        gaps[order[index % count]] += 1
    return gaps


def _pack_region(tokenizer, records, token_budget, rng):
    record_ids = [_encode(tokenizer, _record_text(record)) for record in records]
    filler_budget = token_budget - sum(map(len, record_ids))
    gaps = _allocate_gaps(filler_budget, len(records) + 1, rng)
    filler_text = _make_filler_text(rng)
    filler_ids = _encode(tokenizer, filler_text)
    while len(filler_ids) < filler_budget:
        filler_text += _make_filler_text(rng)
        filler_ids = _encode(tokenizer, filler_text)
    result = []
    cursor = 0
    for index, encoded_record in enumerate(record_ids):
        next_cursor = cursor + gaps[index]
        result.extend(filler_ids[cursor:next_cursor])
        result.extend(encoded_record)
        cursor = next_cursor
    result.extend(filler_ids[cursor:cursor + gaps[-1]])
    if len(result) != token_budget:
        raise AssertionError(f"Packed {len(result)} tokens, expected {token_budget}")
    return result, filler_text


def materialize_scenario(tokenizer, scenario, context_tokens=CONTEXT_TOKENS):
    validate_semantic_scenario(scenario)
    question = f"What is the code for {scenario['target_key']}?"
    messages = [
        {"role": "system", "content": SYSTEM_INSTRUCTION},
        {
            "role": "user",
            "content": f"<DOC>\n{PLACEHOLDER}\n</DOC>\n{question}",
        },
    ]
    template = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    if template.count(PLACEHOLDER) != 1:
        raise ValueError("Chat template did not preserve the document placeholder exactly once")
    prefix_text, suffix_text = template.split(PLACEHOLDER)
    prefix_ids = _encode(tokenizer, prefix_text)
    suffix_ids = _encode(tokenizer, suffix_text)
    if len(suffix_ids) > FINAL_QUERY_WINDOW:
        raise ValueError(
            f"Question/template suffix is {len(suffix_ids)} tokens; maximum is "
            f"{FINAL_QUERY_WINDOW}"
        )
    document_budget = context_tokens - len(prefix_ids) - len(suffix_ids)
    target_record = next(record for record in scenario["records"] if record["is_target"])
    distractors = [record for record in scenario["records"] if not record["is_target"]]
    target_ids = _encode(tokenizer, _record_text(target_record))
    target_start = round(scenario["depth"] * document_budget)
    after_budget = document_budget - target_start - len(target_ids)
    if target_start <= 0 or after_budget <= 0:
        raise ValueError("Target depth leaves no room on one side of the target record")
    rng = random.Random(scenario["seed"] + 17)
    rng.shuffle(distractors)
    before_count = round(scenario["depth"] * len(distractors))
    before_records = distractors[:before_count]
    after_records = distractors[before_count:]
    before_ids, before_filler = _pack_region(
        tokenizer, before_records, target_start, random.Random(scenario["seed"] + 31),
    )
    after_ids, after_filler = _pack_region(
        tokenizer, after_records, after_budget, random.Random(scenario["seed"] + 47),
    )
    document_ids = before_ids + target_ids + after_ids
    input_ids = prefix_ids + document_ids + suffix_ids
    if len(document_ids) != document_budget or len(input_ids) != context_tokens:
        raise AssertionError("Prompt materialization failed the exact-length gate")
    answer = scenario["target_code"]
    leak_text = SYSTEM_INSTRUCTION + suffix_text + before_filler + after_filler
    if answer.casefold() in leak_text.casefold():
        raise AssertionError("Target answer leaked outside its record")
    document_text = tokenizer.decode(document_ids, skip_special_tokens=False)
    extracted_document_codes = [code.upper() for code in CODE_PATTERN.findall(document_text)]
    expected_codes = {record["code"] for record in scenario["records"]}
    if len(extracted_document_codes) != len(scenario["records"]):
        raise AssertionError("A record was truncated or an extra code appeared in the document")
    if set(extracted_document_codes) != expected_codes:
        raise AssertionError("Materialized document codes differ from semantic records")
    if extracted_document_codes.count(answer) != 1:
        raise AssertionError("Target answer must occur exactly once in the document")
    realized_depth = target_start / document_budget
    if abs(realized_depth - scenario["depth"]) > 1 / document_budget:
        raise AssertionError("Target token depth exceeds one-token rounding tolerance")
    return {
        "scenario_id": scenario["scenario_id"],
        "depth": scenario["depth"],
        "target_key": scenario["target_key"],
        "target_code": answer,
        "record_count": len(scenario["records"]),
        "context_tokens": context_tokens,
        "document_tokens": document_budget,
        "target_start_document_token": target_start,
        "target_start_input_token": len(prefix_ids) + target_start,
        "target_record_tokens": len(target_ids),
        "realized_depth": realized_depth,
        "query_window_tokens": len(suffix_ids),
        "input_ids_sha256": token_ids_hash(input_ids),
        "input_ids": input_ids,
    }


def materialize_prompt_manifest(
    tokenizer,
    semantic_manifest,
    tokenizer_id,
    tokenizer_revision,
    context_tokens=CONTEXT_TOKENS,
    include_input_ids=True,
):
    unhashed = {key: value for key, value in semantic_manifest.items() if key != "semantic_sha256"}
    if semantic_manifest.get("semantic_sha256") != canonical_hash(unhashed):
        raise ValueError("Semantic manifest hash is missing or invalid")
    prompts = [
        materialize_scenario(tokenizer, scenario, context_tokens=context_tokens)
        for scenario in semantic_manifest["scenarios"]
    ]
    prompt_set_sha256 = canonical_hash(
        [
            {"scenario_id": prompt["scenario_id"], "sha256": prompt["input_ids_sha256"]}
            for prompt in prompts
        ]
    )
    if not include_input_ids:
        prompts = [
            {key: value for key, value in prompt.items() if key != "input_ids"}
            for prompt in prompts
        ]
    return {
        "manifest_schema": MANIFEST_SCHEMA,
        "generator_version": GENERATOR_VERSION,
        "status": semantic_manifest["status"],
        "semantic_sha256": semantic_manifest["semantic_sha256"],
        "prompt_set_sha256": prompt_set_sha256,
        "tokenizer_id": tokenizer_id,
        "tokenizer_revision": tokenizer_revision,
        "context_tokens": context_tokens,
        "query_window_limit": FINAL_QUERY_WINDOW,
        "prompts": prompts,
    }


def score_response(response, target_code):
    if CODE_PATTERN.fullmatch(target_code) is None:
        raise ValueError(f"Invalid target code: {target_code}")
    extracted = [code.upper() for code in CODE_PATTERN.findall(response)]
    unique_codes = list(dict.fromkeys(extracted))
    target = target_code.upper()
    retrieval_correct = len(unique_codes) == 1 and unique_codes[0] == target
    exact_format = response.strip() == target_code
    legacy_target_anywhere = target.casefold() in response.casefold()
    if retrieval_correct:
        failure_category = "pass"
    elif len(unique_codes) > 1:
        failure_category = "multiple_code"
    elif len(unique_codes) == 1:
        failure_category = "wrong_code"
    elif PARTIAL_CODE_PATTERN.search(response):
        failure_category = "truncated_code"
    else:
        failure_category = "no_code"
    return {
        "retrieval_correct": retrieval_correct,
        "format_correct": exact_format,
        "legacy_target_anywhere": legacy_target_anywhere,
        "failure_category": failure_category,
        "extracted_codes": extracted,
        "unique_extracted_codes": unique_codes,
    }


assert score_response("A123-K45", "A123-K45")["retrieval_correct"]
assert score_response("A123-K", "A123-K45")["failure_category"] == "truncated_code"
assert score_response("none", "A123-K45")["failure_category"] == "no_code"
assert not score_response("A123-K45 B234-L56", "A123-K45")["retrieval_correct"]
print("Frozen generator and strict scorer defined")

## 4. Mandatory Frozen-Hash Gate and Experiment Fingerprint

This cell regenerates all 60 semantic scenarios and all 60 Llama token-ID prompts. It aborts before any model load unless both accepted hashes match. It also freezes the 64-token teacher-forced energy trace and writes only newly generated manifests.

In [ ]:
def atomic_write_text(path, text):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + f".{os.getpid()}.tmp")
    temporary.write_text(text, encoding="utf-8")
    os.replace(temporary, path)


def atomic_write_json(path, value):
    atomic_write_text(path, json.dumps(value, indent=2, ensure_ascii=True) + "\n")


tokenizer = AutoTokenizer.from_pretrained(
    MODEL_AWQ, revision=MODEL_AWQ_REVISION, trust_remote_code=True,
)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

semantic_manifest = build_semantic_manifest()
expected_scenario_ids = [
    f"d{int(depth * 100):02d}-s{index:02d}"
    for depth in DEPTHS
    for index in range(SCENARIOS_PER_DEPTH)
]
assert [row["scenario_id"] for row in semantic_manifest["scenarios"]] == expected_scenario_ids
assert len(semantic_manifest["scenarios"]) == 60
assert semantic_manifest["semantic_sha256"] == EXPECTED_SEMANTIC_SHA256, (
    semantic_manifest["semantic_sha256"], EXPECTED_SEMANTIC_SHA256
)

prompt_manifest = materialize_prompt_manifest(
    tokenizer,
    semantic_manifest,
    tokenizer_id=MODEL_AWQ,
    tokenizer_revision=MODEL_AWQ_REVISION,
    context_tokens=CONTEXT_TOKENS,
    include_input_ids=True,
)
assert [row["scenario_id"] for row in prompt_manifest["prompts"]] == expected_scenario_ids
assert prompt_manifest["prompt_set_sha256"] == EXPECTED_PROMPT_SET_SHA256, (
    prompt_manifest["prompt_set_sha256"], EXPECTED_PROMPT_SET_SHA256
)
assert all(row["context_tokens"] == CONTEXT_TOKENS for row in prompt_manifest["prompts"])
assert all(len(row["input_ids"]) == CONTEXT_TOKENS for row in prompt_manifest["prompts"])
assert all(row["record_count"] == RECORD_COUNT for row in prompt_manifest["prompts"])
assert all(row["query_window_tokens"] <= FINAL_QUERY_WINDOW for row in prompt_manifest["prompts"])
prompts = {row["scenario_id"]: row for row in prompt_manifest["prompts"]}
assert len(prompts) == 60

TRACE_TEXT = (
    "A controlled decode trace keeps the generated workload identical across every "
    "compression configuration. This prevents early end tokens from changing the amount "
    "of measured work and supports a fair energy comparison. "
) * 4
TRACE_IDS = tokenizer.encode(TRACE_TEXT, add_special_tokens=False)[:FIXED_DECODE_TOKENS]
assert len(TRACE_IDS) == FIXED_DECODE_TOKENS
TRACE_IDS_SHA256 = token_ids_hash(TRACE_IDS)
TRACE_TEXT_SHA256 = hashlib.sha256(TRACE_TEXT.encode("utf-8")).hexdigest()

FINGERPRINT_DATA = {
    "schema_version": SCHEMA_VERSION,
    "model_fp16": MODEL_FP16,
    "model_fp16_revision": MODEL_FP16_REVISION,
    "model_awq": MODEL_AWQ,
    "model_awq_revision": MODEL_AWQ_REVISION,
    "transformers": "5.12.1",
    "turboquant": "0.2.0",
    "turboquant_source_sha256": EXPECTED_TQ_SOURCE_SHA256,
    "generator_version": GENERATOR_VERSION,
    "semantic_sha256": EXPECTED_SEMANTIC_SHA256,
    "prompt_set_sha256": EXPECTED_PROMPT_SET_SHA256,
    "context_tokens": CONTEXT_TOKENS,
    "depths": list(DEPTHS),
    "scenarios_per_depth": SCENARIOS_PER_DEPTH,
    "record_count": RECORD_COUNT,
    "quality_variants": list(QUALITY_VARIANTS),
    "quality_max_new_tokens": MAX_NEW_TOKENS,
    "average_budget": DEFAULT_AVERAGE_BUDGET,
    "tq_bits": DEFAULT_TQ_BITS,
    "window": WINDOW,
    "pool_kernel": POOL_KERNEL,
    "pyramid_beta": PYRAMID_BETA,
    "parity_atol": PARITY_ATOL,
    "parity_rtol": PARITY_RTOL,
    "energy_blocks": MEASURED_BLOCKS,
    "fixed_decode_tokens": FIXED_DECODE_TOKENS,
    "trace_text_sha256": TRACE_TEXT_SHA256,
}
EXPERIMENT_FINGERPRINT = canonical_hash(FINGERPRINT_DATA)[:16]
assert EXPERIMENT_FINGERPRINT == EXPECTED_FINGERPRINT, (
    EXPERIMENT_FINGERPRINT, EXPECTED_FINGERPRINT
)

atomic_write_json(SEMANTIC_MANIFEST_JSON, semantic_manifest)
atomic_write_json(PROMPT_MANIFEST_JSON, prompt_manifest)
environment = {
    **FINGERPRINT_DATA,
    "fingerprint": EXPERIMENT_FINGERPRINT,
    "session_id": SESSION_ID,
    "gpu": GPU.name,
    "vram_gb": round(GPU.total_memory / 2**30, 2),
    "torch": torch.__version__,
    "cuda_runtime": torch.version.cuda,
    "gptqmodel": importlib.metadata.version("gptqmodel"),
    "artifact_path": SAVE_DIR.name,
    "label_policy": {
        "C2": "AWQ + custom pyramidal eviction",
        "C3-Ref": "AWQ + turboquant==0.2.0 numerical reference path",
        "C4-Ref": "AWQ + sequential eviction then TQ-Ref",
        "TQ-Ref_boundary": "not packed, fused, or physically 4-bit",
    },
}
atomic_write_json(ENVIRONMENT_JSON, environment)
print("Frozen hashes and experiment fingerprint passed:", EXPERIMENT_FINGERPRINT)

## 5. Atomic Row Persistence and Resumption

Every completed case is first committed as one atomic JSON shard. The CSV is an atomic snapshot rebuilt from validated shards. Existing shards are accepted only if their schema, fingerprint, key, frozen prompt hash, and structural gate are valid.

In [ ]:
def safe_component(value):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value))


def shard_path(row_dir, key_values):
    return row_dir / ("__".join(safe_component(value) for value in key_values) + ".json")


def read_valid_shards(row_dir, key_columns, allowed_variants, require_prompt=False):
    rows = []
    keys = set()
    for path in sorted(row_dir.glob("*.json")):
        row = json.loads(path.read_text(encoding="utf-8"))
        assert row["fingerprint"] == EXPERIMENT_FINGERPRINT, path
        assert row["schema_version"] == SCHEMA_VERSION, path
        assert row["variant"] in allowed_variants, path
        assert row["structural_pass"] is True, path
        key = tuple(row[column] for column in key_columns)
        assert key not in keys, (path, key)
        assert path == shard_path(row_dir, key), (path, key)
        if require_prompt:
            prompt = prompts[row["scenario_id"]]
            assert row["input_ids_sha256"] == prompt["input_ids_sha256"], path
        keys.add(key)
        rows.append(row)
    return rows, keys


def refresh_csv(row_dir, csv_path, key_columns, allowed_variants, require_prompt=False):
    rows, keys = read_valid_shards(
        row_dir, key_columns, allowed_variants, require_prompt=require_prompt,
    )
    frame = pd.DataFrame(rows)
    if len(frame):
        frame = frame.sort_values(list(key_columns)).reset_index(drop=True)
    temporary = csv_path.with_suffix(csv_path.suffix + f".{os.getpid()}.tmp")
    frame.to_csv(temporary, index=False)
    os.replace(temporary, csv_path)
    return rows, keys


def commit_row(row_dir, csv_path, row, key_columns, allowed_variants, require_prompt=False):
    key = tuple(row[column] for column in key_columns)
    destination = shard_path(row_dir, key)
    if destination.exists():
        existing = json.loads(destination.read_text(encoding="utf-8"))
        assert existing == row, f"Conflicting completed shard: {destination}"
    else:
        atomic_write_json(destination, row)
    refresh_csv(
        row_dir, csv_path, key_columns, allowed_variants, require_prompt=require_prompt,
    )


def require_frozen_gate():
    environment = json.loads(ENVIRONMENT_JSON.read_text(encoding="utf-8"))
    assert environment["fingerprint"] == EXPERIMENT_FINGERPRINT
    assert environment["semantic_sha256"] == EXPECTED_SEMANTIC_SHA256
    assert environment["prompt_set_sha256"] == EXPECTED_PROMPT_SET_SHA256
    assert semantic_manifest["semantic_sha256"] == EXPECTED_SEMANTIC_SHA256
    assert prompt_manifest["prompt_set_sha256"] == EXPECTED_PROMPT_SET_SHA256


def require_validation_gate():
    require_frozen_gate()
    record = json.loads(VALIDATION_JSON.read_text(encoding="utf-8"))
    assert record["passed"] is True
    assert record["fingerprint"] == EXPERIMENT_FINGERPRINT
    assert record["semantic_sha256"] == EXPECTED_SEMANTIC_SHA256
    assert record["prompt_set_sha256"] == EXPECTED_PROMPT_SET_SHA256


print("Atomic resumability helpers ready")

## 6. Models, Matched Wrapper, and Cache Validation

C1-W uses the wrapper even with both compression flags off. Prefill attention always sees all 4096 tokens. C4-Ref selects the layer budget first and passes only that selected K/V to `TurboQuantCache.update`. Decode calls record active use of the returned cache values.

In [ ]:
def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()


def release_model():
    previous = globals().pop("model", None)
    if previous is not None:
        del previous
    clear_gpu()


def validate_model_architecture(model):
    config = model.config
    assert config.model_type == "llama"
    assert config.num_hidden_layers == NUM_LAYERS
    assert config.num_attention_heads == NUM_Q_HEADS
    assert config.num_key_value_heads == NUM_KV_HEADS
    assert getattr(config, "head_dim", config.hidden_size // NUM_Q_HEADS) == HEAD_DIM
    assert config.vocab_size == len(tokenizer)


def load_fp16():
    require_frozen_gate()
    assert LlamaAttention.forward is ORIGINAL_LLAMA_FORWARD
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_FP16,
        revision=MODEL_FP16_REVISION,
        dtype=torch.float16,
        device_map={"": 0},
        attn_implementation="sdpa",
        trust_remote_code=True,
    ).eval()
    validate_model_architecture(model)
    return model


def load_awq():
    require_frozen_gate()
    assert LlamaAttention.forward is ORIGINAL_LLAMA_FORWARD
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_AWQ,
        revision=MODEL_AWQ_REVISION,
        device_map={"": 0},
        attn_implementation="sdpa",
        trust_remote_code=True,
    ).eval()
    validate_model_architecture(model)
    raw = model.config.quantization_config
    quant = raw.to_dict() if hasattr(raw, "to_dict") else dict(raw)
    assert quant["quant_method"] == "awq"
    assert quant["bits"] == 4 and quant["group_size"] == 128
    return model


def build_pyramid_budgets(
    layers=NUM_LAYERS,
    average_budget=DEFAULT_AVERAGE_BUDGET,
    window=WINDOW,
    beta=PYRAMID_BETA,
):
    average_budget = int(average_budget)
    assert layers > 1 and average_budget > window and beta > 0
    old_average = average_budget - window
    last = round(old_average / beta)
    first = 2 * old_average - last
    budgets = (np.linspace(first, last, layers).round().astype(int) + window).tolist()
    assert len(budgets) == layers and int(np.mean(budgets)) == average_budget
    assert min(budgets) >= window and all(a >= b for a, b in zip(budgets, budgets[1:]))
    return budgets


PYRAMID_BUDGETS = build_pyramid_budgets()
VARIANT_PATHS = {
    "C0": {"wrapper": False, "eviction": False, "tq": False},
    "C1-S": {"wrapper": False, "eviction": False, "tq": False},
    "C1-W": {"wrapper": True, "eviction": False, "tq": False},
    "C2": {"wrapper": True, "eviction": True, "tq": False},
    "C3-Ref": {"wrapper": True, "eviction": False, "tq": True},
    "C4-Ref": {"wrapper": True, "eviction": True, "tq": True},
}
PKV_LOG = []
CACHE_UPDATE_LOG = []
ACTIVE_EVICTION = False
ORIGINAL_LLAMA_FORWARD = LlamaAttention.forward


def select_pyramid_kv(query_states, key_states, value_states, layer_idx, groups, scaling):
    cache_len = key_states.shape[-2]
    budget = min(PYRAMID_BUDGETS[layer_idx], cache_len)
    if cache_len <= budget:
        return key_states, value_states
    obs = min(WINDOW, cache_len)
    old_len = cache_len - obs
    old_keep = max(budget - obs, 0)
    recent_query = query_states[:, :, -obs:, :]
    repeated_key = repeat_kv(key_states, groups) if groups > 1 else key_states
    scores = torch.matmul(recent_query, repeated_key.transpose(2, 3)) * scaling
    causal = torch.triu(
        torch.full((obs, obs), float("-inf"), device=scores.device, dtype=scores.dtype),
        diagonal=1,
    )
    scores[:, :, :, -obs:] += causal[None, None, :, :]
    probabilities = F.softmax(scores, dim=-1, dtype=torch.float32)
    prefix_scores = probabilities[:, :, :, :old_len].sum(dim=-2)
    prefix_scores = prefix_scores.reshape(
        prefix_scores.shape[0], key_states.shape[1], groups, old_len,
    ).mean(dim=2)
    prefix_scores = F.avg_pool1d(
        prefix_scores, kernel_size=POOL_KERNEL, stride=1, padding=POOL_KERNEL // 2,
    )
    topk = prefix_scores.topk(old_keep, dim=-1).indices.sort(dim=-1).values
    gather_index = topk.unsqueeze(-1).expand(-1, -1, -1, key_states.shape[-1])
    selected_key = torch.cat(
        [key_states[:, :, :old_len, :].gather(2, gather_index), key_states[:, :, -obs:, :]],
        dim=2,
    ).contiguous()
    selected_value = torch.cat(
        [value_states[:, :, :old_len, :].gather(2, gather_index), value_states[:, :, -obs:, :]],
        dim=2,
    ).contiguous()
    PKV_LOG.append({
        "layer": int(layer_idx),
        "before": int(cache_len),
        "after": int(selected_key.shape[-2]),
        "budget": int(budget),
        "head_selection_differs": bool((topk[:, 1:] != topk[:, :-1]).any().item()),
    })
    return selected_key, selected_value


def cache_wrapper_forward(
    self,
    hidden_states,
    position_embeddings=None,
    attention_mask=None,
    past_key_values=None,
    **kwargs,
):
    input_shape = hidden_states.shape[:-1]
    hidden_shape = (*input_shape, -1, self.head_dim)
    query_states = self.q_proj(hidden_states).view(hidden_shape).transpose(1, 2)
    key_states = self.k_proj(hidden_states).view(hidden_shape).transpose(1, 2)
    value_states = self.v_proj(hidden_states).view(hidden_shape).transpose(1, 2)
    cos, sin = position_embeddings
    query_states, key_states = apply_rotary_pos_emb(query_states, key_states, cos, sin)
    is_prefill = (
        past_key_values is not None
        and past_key_values.get_seq_length(self.layer_idx) == 0
    )
    if past_key_values is not None and not is_prefill:
        input_tokens = int(key_states.shape[-2])
        key_states, value_states = past_key_values.update(
            key_states, value_states, self.layer_idx,
        )
        CACHE_UPDATE_LOG.append({
            "phase": "decode",
            "layer": int(self.layer_idx),
            "input_tokens": input_tokens,
            "returned_tokens": int(key_states.shape[-2]),
            "cache_type": type(past_key_values).__name__,
        })
    attention_tokens = int(key_states.shape[-2])
    attention_interface = ALL_ATTENTION_FUNCTIONS.get_interface(
        self.config._attn_implementation, eager_attention_forward,
    )
    attention_output, attention_weights = attention_interface(
        self,
        query_states,
        key_states,
        value_states,
        attention_mask,
        dropout=0.0,
        scaling=self.scaling,
        **kwargs,
    )
    if is_prefill:
        stored_key, stored_value = key_states, value_states
        if ACTIVE_EVICTION:
            stored_key, stored_value = select_pyramid_kv(
                query_states,
                key_states,
                value_states,
                self.layer_idx,
                self.num_key_value_groups,
                self.scaling,
            )
        update_input_tokens = int(stored_key.shape[-2])
        returned_key, returned_value = past_key_values.update(
            stored_key, stored_value, self.layer_idx,
        )
        CACHE_UPDATE_LOG.append({
            "phase": "prefill",
            "layer": int(self.layer_idx),
            "attention_tokens": attention_tokens,
            "update_input_tokens": update_input_tokens,
            "returned_tokens": int(returned_key.shape[-2]),
            "cache_type": type(past_key_values).__name__,
        })
        del returned_key, returned_value
    attention_output = attention_output.reshape(*input_shape, -1).contiguous()
    return self.o_proj(attention_output), attention_weights


def enable_wrapper(use_eviction):
    global ACTIVE_EVICTION
    assert LlamaAttention.forward is ORIGINAL_LLAMA_FORWARD
    PKV_LOG.clear()
    CACHE_UPDATE_LOG.clear()
    ACTIVE_EVICTION = bool(use_eviction)
    LlamaAttention.forward = cache_wrapper_forward


def disable_wrapper():
    global ACTIVE_EVICTION
    LlamaAttention.forward = ORIGINAL_LLAMA_FORWARD
    ACTIVE_EVICTION = False
    PKV_LOG.clear()
    CACHE_UPDATE_LOG.clear()
    assert LlamaAttention.forward is ORIGINAL_LLAMA_FORWARD


def new_cache(variant):
    return TurboQuantCache(bits=DEFAULT_TQ_BITS) if VARIANT_PATHS[variant]["tq"] else DynamicCache()


def cache_lengths(cache):
    lengths = []
    for layer in cache.layers:
        if hasattr(layer, "_total_len"):
            lengths.append(int(layer._total_len))
        else:
            lengths.append(int(layer.get_seq_length()))
    return lengths


def named_cache_tensors(cache):
    for layer_index, layer in enumerate(cache.layers):
        for name, value in vars(layer).items():
            if isinstance(value, torch.Tensor):
                yield layer_index, name, value
            elif isinstance(value, Mapping):
                for nested_name, nested_value in value.items():
                    if isinstance(nested_value, torch.Tensor):
                        yield layer_index, f"{name}.{nested_name}", nested_value
            elif isinstance(value, (list, tuple)):
                for nested_index, nested_value in enumerate(value):
                    if isinstance(nested_value, torch.Tensor):
                        yield layer_index, f"{name}.{nested_index}", nested_value


def storage_bytes(tensors):
    seen = set()
    total = 0
    for tensor in tensors:
        if not isinstance(tensor, torch.Tensor) or tensor.numel() == 0:
            continue
        storage = tensor.untyped_storage()
        key = (str(tensor.device), int(storage.data_ptr()), int(storage.nbytes()))
        if key not in seen:
            seen.add(key)
            total += int(storage.nbytes())
    return total


def cache_storage_metrics(cache, prefix):
    named = list(named_cache_tensors(cache))
    all_tensors = [tensor for _, _, tensor in named]
    indices = [tensor for _, name, tensor in named if name in {"_key_indices", "_value_indices"}]
    norms = [tensor for _, name, tensor in named if name in {"_key_norms", "_value_norms"}]
    residuals = [tensor for _, name, tensor in named if name in {"_residual_keys", "_residual_values"}]
    materialized = [tensor for _, name, tensor in named if name in {"keys", "values"}]
    payload = indices + norms + residuals
    package_bytes = 0
    if callable(getattr(cache, "memory_usage_bytes", None)):
        package_bytes = int(cache.memory_usage_bytes().get("total_bytes", 0))
    return {
        f"{prefix}_cache_live_bytes": storage_bytes(all_tensors),
        f"{prefix}_tq_index_bytes": storage_bytes(indices),
        f"{prefix}_tq_norm_bytes": storage_bytes(norms),
        f"{prefix}_tq_residual_bytes": storage_bytes(residuals),
        f"{prefix}_tq_payload_bytes": storage_bytes(payload),
        f"{prefix}_tq_materialized_bytes": storage_bytes(materialized),
        f"{prefix}_tq_package_reported_bytes": package_bytes,
        f"{prefix}_tensor_count": len(all_tensors),
    }


TQ_STATE_NAMES = {
    "_key_indices", "_value_indices", "_key_norms", "_value_norms",
    "_residual_keys", "_residual_values",
}


def assert_no_tq_state(cache):
    assert isinstance(cache, DynamicCache) and not isinstance(cache, TurboQuantCache)
    for _, name, _ in named_cache_tensors(cache):
        assert name not in TQ_STATE_NAMES


def assert_no_compression_state(cache):
    assert ACTIVE_EVICTION is False
    assert not PKV_LOG
    assert_no_tq_state(cache)


def validate_cache_path(cache, variant, prefill_lengths, decode_tokens, update_log):
    path = VARIANT_PATHS[variant]
    assert len(cache.layers) == NUM_LAYERS
    expected_prefill = (
        [min(CONTEXT_TOKENS, budget) for budget in PYRAMID_BUDGETS]
        if path["eviction"] else [CONTEXT_TOKENS] * NUM_LAYERS
    )
    assert prefill_lengths == expected_prefill
    final_lengths = cache_lengths(cache)
    assert final_lengths == [length + decode_tokens for length in expected_prefill]
    prefill_updates = [row for row in update_log if row["phase"] == "prefill"]
    decode_updates = [row for row in update_log if row["phase"] == "decode"]
    if path["wrapper"]:
        assert len(prefill_updates) == NUM_LAYERS
        assert all(row["attention_tokens"] == CONTEXT_TOKENS for row in prefill_updates)
        assert [row["update_input_tokens"] for row in prefill_updates] == expected_prefill
        assert len(decode_updates) == NUM_LAYERS * decode_tokens
        assert all(row["input_tokens"] == 1 for row in decode_updates)
        assert sorted({row["layer"] for row in decode_updates}) == (
            list(range(NUM_LAYERS)) if decode_tokens else []
        )
    else:
        assert not update_log
    if path["eviction"]:
        assert len(PKV_LOG) == NUM_LAYERS
        assert sorted(row["layer"] for row in PKV_LOG) == list(range(NUM_LAYERS))
        assert all(row["before"] == CONTEXT_TOKENS for row in PKV_LOG)
        assert [row["after"] for row in sorted(PKV_LOG, key=lambda row: row["layer"])] == expected_prefill
    else:
        assert not PKV_LOG
    quantized_layers = 0
    if path["tq"]:
        assert isinstance(cache, TurboQuantCache) and cache.bits == DEFAULT_TQ_BITS
        for layer, total_length in zip(cache.layers, final_lengths):
            assert layer.bits == DEFAULT_TQ_BITS
            quantized_tokens = max(total_length - layer.residual_len, 0)
            residual_tokens = min(total_length, layer.residual_len)
            index_shape = (1, NUM_KV_HEADS, quantized_tokens, HEAD_DIM)
            norm_shape = (1, NUM_KV_HEADS, quantized_tokens, 1)
            if quantized_tokens:
                quantized_layers += 1
                assert tuple(layer._key_indices.shape) == index_shape
                assert tuple(layer._value_indices.shape) == index_shape
                assert tuple(layer._key_norms.shape) == norm_shape
                assert tuple(layer._value_norms.shape) == norm_shape
                for indices in (layer._key_indices, layer._value_indices):
                    assert indices.dtype == torch.uint8
                    assert int(indices.max()) < 2 ** DEFAULT_TQ_BITS
                for norms in (layer._key_norms, layer._value_norms):
                    assert torch.isfinite(norms).all() and (norms >= 0).all()
            else:
                assert layer._key_indices.numel() == layer._value_indices.numel() == 0
            assert layer._residual_keys.shape[-2] == residual_tokens
            assert layer._residual_values.shape[-2] == residual_tokens
            assert layer.keys.shape[-2] == total_length
            assert layer.values.shape[-2] == total_length
            assert torch.isfinite(layer.keys).all() and torch.isfinite(layer.values).all()
        assert quantized_layers == sum(length > 128 for length in final_lengths)
        metrics = cache_storage_metrics(cache, "validation")
        assert metrics["validation_tq_index_bytes"] > 0
        assert metrics["validation_tq_norm_bytes"] > 0
        assert metrics["validation_tq_residual_bytes"] > 0
        assert metrics["validation_tq_materialized_bytes"] > 0
    else:
        assert_no_tq_state(cache)
        if not path["eviction"]:
            assert_no_compression_state(cache)
    return {
        "structural_pass": True,
        "quantized_layers": quantized_layers,
        "evicted_layers": NUM_LAYERS if path["eviction"] else 0,
        "cache_min_prefill": min(prefill_lengths),
        "cache_max_prefill": max(prefill_lengths),
        "cache_min_after": min(final_lengths),
        "cache_max_after": max(final_lengths),
        "active_decode_cache_updates": len(decode_updates),
    }


print("Wrapper, model loaders, cache audit, and active-use logging ready")

## 7. C1-S vs C1-W Parity and Compression-Path Gates

Predeclared parity tolerances are `atol=0.005` and `rtol=0.005`. Three fixed prompts test final-position logits, top-1 agreement, exact greedy token IDs/responses, all 28 cache lengths, and absence of eviction/TQ state. Separate C2/C3-Ref/C4-Ref probes validate eviction order, TQ indices/norms/residuals/materialized values, live-storage accounting, and active cache-backed decode.

In [ ]:
def prompt_tensor(prompt, device):
    ids = torch.tensor([prompt["input_ids"]], dtype=torch.long, device=device)
    actual_hash = token_ids_hash(ids[0].detach().cpu().tolist())
    assert ids.shape == (1, CONTEXT_TOKENS)
    assert actual_hash == prompt["input_ids_sha256"]
    return ids, actual_hash


def run_greedy_path(model, variant, prompt, max_new_tokens=MAX_NEW_TOKENS):
    path = VARIANT_PATHS[variant]
    if path["wrapper"]:
        enable_wrapper(path["eviction"])
    else:
        assert LlamaAttention.forward is ORIGINAL_LLAMA_FORWARD
        PKV_LOG.clear()
        CACHE_UPDATE_LOG.clear()
    cache = new_cache(variant)
    ids, actual_hash = prompt_tensor(prompt, model.device)
    generated = []
    try:
        with torch.inference_mode():
            output = model(
                input_ids=ids, past_key_values=cache, use_cache=True, logits_to_keep=1,
            )
            prefill_lengths = cache_lengths(cache)
            prefill_logits = output.logits[0, -1].float().detach().cpu()
            next_token = output.logits[:, -1, :].argmax(-1, keepdim=True)
            for step in range(max_new_tokens):
                generated.append(next_token)
                if next_token.item() in np.atleast_1d(tokenizer.eos_token_id):
                    break
                if step == max_new_tokens - 1:
                    break
                absolute_position = CONTEXT_TOKENS + step
                position = torch.tensor([[absolute_position]], device=model.device)
                cache_position = torch.tensor([absolute_position], device=model.device)
                output = model(
                    input_ids=next_token,
                    position_ids=position,
                    cache_position=cache_position,
                    past_key_values=cache,
                    use_cache=True,
                    logits_to_keep=1,
                )
                next_token = output.logits[:, -1, :].argmax(-1, keepdim=True)
        output_ids = torch.cat(generated, dim=1)[0].tolist()
        update_log = list(CACHE_UPDATE_LOG)
        structure = validate_cache_path(
            cache, variant, prefill_lengths, max(0, len(output_ids) - 1), update_log,
        )
        storage = cache_storage_metrics(cache, "generation_final")
        return {
            "input_ids_sha256": actual_hash,
            "prefill_logits": prefill_logits,
            "prefill_top1": int(prefill_logits.argmax()),
            "response": tokenizer.decode(output_ids, skip_special_tokens=True),
            "output_token_ids": output_ids,
            "prefill_lengths": prefill_lengths,
            "final_lengths": cache_lengths(cache),
            "structure": structure,
            "storage": storage,
        }
    finally:
        if path["wrapper"]:
            disable_wrapper()
        del cache, ids
        clear_gpu()


def run_path_probe(model, variant, prompt):
    path = VARIANT_PATHS[variant]
    enable_wrapper(path["eviction"])
    cache = new_cache(variant)
    ids, actual_hash = prompt_tensor(prompt, model.device)
    answer_ids = tokenizer.encode(prompt["target_code"], add_special_tokens=False)
    assert tokenizer.decode(answer_ids, skip_special_tokens=True) == prompt["target_code"]
    try:
        with torch.inference_mode():
            output = model(
                input_ids=ids, past_key_values=cache, use_cache=True, logits_to_keep=1,
            )
            prefill_lengths = cache_lengths(cache)
            prefill_logits = output.logits[0, -1].float().detach().cpu()
            token = torch.tensor([[answer_ids[0]]], dtype=torch.long, device=model.device)
            position = torch.tensor([[CONTEXT_TOKENS]], device=model.device)
            cache_position = torch.tensor([CONTEXT_TOKENS], device=model.device)
            model(
                input_ids=token,
                position_ids=position,
                cache_position=cache_position,
                past_key_values=cache,
                use_cache=True,
                logits_to_keep=1,
            )
        update_log = list(CACHE_UPDATE_LOG)
        structure = validate_cache_path(cache, variant, prefill_lengths, 1, update_log)
        storage = cache_storage_metrics(cache, "probe_final")
        return {
            "variant": variant,
            "input_ids_sha256": actual_hash,
            "prefill_logits": prefill_logits,
            "prefill_top1": int(prefill_logits.argmax()),
            "prefill_lengths": prefill_lengths,
            "final_lengths": cache_lengths(cache),
            "structure": structure,
            "storage": storage,
            "prefill_update_inputs": [
                row["update_input_tokens"] for row in update_log if row["phase"] == "prefill"
            ],
        }
    finally:
        disable_wrapper()
        del cache, ids
        clear_gpu()


require_frozen_gate()
VALIDATION_JSON.unlink(missing_ok=True)
model = load_awq()
parity_scenarios = ("d10-s00", "d50-s00", "d90-s00")
parity_rows = []
path_rows = []
validation = {"passed": False}
try:
    for scenario_id in parity_scenarios:
        prompt = prompts[scenario_id]
        stock = run_greedy_path(model, "C1-S", prompt)
        wrapped = run_greedy_path(model, "C1-W", prompt)
        allclose = torch.allclose(
            stock["prefill_logits"], wrapped["prefill_logits"],
            atol=PARITY_ATOL, rtol=PARITY_RTOL,
        )
        max_abs = float((stock["prefill_logits"] - wrapped["prefill_logits"]).abs().max())
        top1_agree = stock["prefill_top1"] == wrapped["prefill_top1"]
        token_ids_equal = stock["output_token_ids"] == wrapped["output_token_ids"]
        responses_equal = stock["response"] == wrapped["response"]
        cache_lengths_equal = (
            stock["prefill_lengths"] == wrapped["prefill_lengths"] == [CONTEXT_TOKENS] * NUM_LAYERS
            and stock["final_lengths"] == wrapped["final_lengths"]
        )
        assert allclose and top1_agree and token_ids_equal and responses_equal and cache_lengths_equal
        parity_rows.append({
            "scenario_id": scenario_id,
            "input_ids_sha256": prompt["input_ids_sha256"],
            "max_abs_logit_difference": max_abs,
            "logits_allclose": allclose,
            "top1_agreement": top1_agree,
            "greedy_token_ids_equal": token_ids_equal,
            "greedy_responses_equal": responses_equal,
            "full_cache_lengths_equal": cache_lengths_equal,
            "absence_of_eviction_quantization_state": True,
            "stock_output_token_ids": stock["output_token_ids"],
            "wrapped_output_token_ids": wrapped["output_token_ids"],
        })

    probe_prompt = prompts["d50-s00"]
    for variant in ("C1-W", "C2", "C3-Ref", "C4-Ref"):
        probe = run_path_probe(model, variant, probe_prompt)
        path_rows.append(probe)
    baseline_logits = path_rows[0]["prefill_logits"]
    for probe in path_rows[1:]:
        assert torch.allclose(
            baseline_logits, probe["prefill_logits"],
            atol=PARITY_ATOL, rtol=PARITY_RTOL,
        ), probe["variant"]
        assert probe["prefill_top1"] == path_rows[0]["prefill_top1"]
        assert probe["structure"]["active_decode_cache_updates"] == NUM_LAYERS
    assert path_rows[2]["storage"]["probe_final_tq_materialized_bytes"] > 0
    assert path_rows[3]["storage"]["probe_final_tq_materialized_bytes"] > 0
    assert path_rows[3]["prefill_update_inputs"] == [
        min(CONTEXT_TOKENS, budget) for budget in PYRAMID_BUDGETS
    ]

    samples = torch.randn(1024, HEAD_DIM, device=model.device, dtype=torch.float16)
    distortion_rows = []
    for bits in (2, 3, 4):
        quantizer = TurboQuantMSE(dim=HEAD_DIM, bits=bits, device=str(model.device))
        indices, norms = quantizer.quantize(samples)
        restored = quantizer.dequantize(indices, norms)
        assert indices.dtype == torch.uint8 and int(indices.max()) < 2 ** bits
        assert torch.isfinite(norms).all() and torch.isfinite(restored).all()
        distortion_rows.append({
            "bits": bits,
            "normalized_mse": float(
                ((restored.float() - samples.float()).square().sum(-1)
                 / samples.float().square().sum(-1).clamp_min(1e-12)).mean()
            ),
            "physical_index_bytes_per_coordinate": indices.element_size(),
        })
    assert all(row["physical_index_bytes_per_coordinate"] == 1 for row in distortion_rows)

    parity_record = {
        "fingerprint": EXPERIMENT_FINGERPRINT,
        "schema_version": SCHEMA_VERSION,
        "atol": PARITY_ATOL,
        "rtol": PARITY_RTOL,
        "passed": True,
        "rows": parity_rows,
    }
    atomic_write_json(PARITY_JSON, parity_record)
    serializable_paths = []
    for probe in path_rows:
        serializable_paths.append({
            key: value for key, value in probe.items() if key != "prefill_logits"
        })
    validation = {
        "passed": True,
        "fingerprint": EXPERIMENT_FINGERPRINT,
        "schema_version": SCHEMA_VERSION,
        "semantic_sha256": EXPECTED_SEMANTIC_SHA256,
        "prompt_set_sha256": EXPECTED_PROMPT_SET_SHA256,
        "parity": parity_record,
        "validated_paths": serializable_paths,
        "tq_distortion": distortion_rows,
        "tq_storage_boundary": (
            "TurboQuant 0.2.0 is a numerical reference path with uint8 indices and "
            "persistent materialized K/V; it is not packed, fused, or physically 4-bit."
        ),
    }
    atomic_write_json(VALIDATION_JSON, validation)
    print("All mandatory parity and cache-path gates passed")
finally:
    disable_wrapper()
    release_model()

assert validation["passed"], "Do not run measurements until every validation gate passes"

## 8. Mandatory Five-Block C1-W 4096-Token Energy Control

This is the only new mandatory Llama energy condition. It uses the matched wrapper with no eviction or TQ state, one unrecorded warm-up, five measured blocks, the v5 fixed 64-token teacher-forced trace, synchronized prefill/decode timing, and approximately 10 Hz NVML sampling. Raw trapezoidal joules are primary.

In [ ]:
def nvml_value(function, *args, default=np.nan):
    try:
        return function(*args)
    except (AttributeError, pynvml.NVMLError):
        return default


def sensor_sample():
    memory = pynvml.nvmlDeviceGetMemoryInfo(NVML_HANDLE)
    return {
        "power_w": float(pynvml.nvmlDeviceGetPowerUsage(NVML_HANDLE) / 1000.0),
        "nvml_used_mb": float(memory.used / 2**20),
        "temperature_c": float(nvml_value(
            pynvml.nvmlDeviceGetTemperature,
            NVML_HANDLE,
            pynvml.NVML_TEMPERATURE_GPU,
        )),
    }


def telemetry_snapshot():
    sample = sensor_sample()
    sample.update({
        "sm_clock_mhz": float(nvml_value(
            pynvml.nvmlDeviceGetClockInfo, NVML_HANDLE, pynvml.NVML_CLOCK_SM,
        )),
        "memory_clock_mhz": float(nvml_value(
            pynvml.nvmlDeviceGetClockInfo, NVML_HANDLE, pynvml.NVML_CLOCK_MEM,
        )),
        "power_limit_w": float(nvml_value(
            pynvml.nvmlDeviceGetEnforcedPowerLimit, NVML_HANDLE,
        )) / 1000.0,
        "throttle_reasons": int(nvml_value(
            pynvml.nvmlDeviceGetCurrentClocksThrottleReasons,
            NVML_HANDLE,
            default=0,
        )),
    })
    return sample


class GPUMonitor:
    def __init__(self, interval_s=0.1):
        self.interval_s = float(interval_s)
        self.samples = []
        self.running = False
        self.stop_event = threading.Event()
        self.lock = threading.Lock()

    def sample(self, elapsed_s=None):
        row = sensor_sample()
        row["elapsed_s"] = float(
            time.perf_counter() - self.t0 if elapsed_s is None else elapsed_s
        )
        with self.lock:
            self.samples.append(row)

    def poll(self):
        deadline = self.t0 + self.interval_s
        while not self.stop_event.wait(max(0.0, deadline - time.perf_counter())):
            try:
                self.sample()
            except pynvml.NVMLError:
                pass
            deadline += self.interval_s

    def start(self):
        assert not self.running
        torch.cuda.synchronize()
        self.samples = []
        self.stop_event.clear()
        self.t0 = time.perf_counter()
        self.sample(elapsed_s=0.0)
        self.running = True
        self.thread = threading.Thread(target=self.poll, daemon=True)
        self.thread.start()

    def stop(self):
        assert self.running
        torch.cuda.synchronize()
        self.stop_event.set()
        self.thread.join(timeout=2)
        if self.thread.is_alive():
            raise RuntimeError("Power monitor thread did not terminate")
        self.sample()
        self.running = False
        with self.lock:
            samples = sorted(self.samples, key=lambda row: row["elapsed_s"])
        assert len(samples) >= 2
        times = np.asarray([row["elapsed_s"] for row in samples], dtype=float)
        powers = np.asarray([row["power_w"] for row in samples], dtype=float)
        assert np.all(np.diff(times) >= 0) and np.isfinite(powers).all()
        duration = float(times[-1] - times[0])
        energy = float(np.trapezoid(powers, times))
        temperatures = [
            row["temperature_c"] for row in samples if np.isfinite(row["temperature_c"])
        ]
        return {
            "energy_j": energy,
            "duration_s": duration,
            "avg_power_w": energy / duration,
            "max_power_w": float(powers.max()),
            "nvml_peak_mb": float(max(row["nvml_used_mb"] for row in samples)),
            "max_temperature_c": float(max(temperatures)) if temperatures else np.nan,
            "n_samples": len(samples),
            "samples": samples,
        }


def measure_idle_telemetry(seconds=10.0):
    torch.cuda.synchronize()
    samples = []
    deadline = time.perf_counter() + seconds
    while True:
        samples.append(sensor_sample())
        if time.perf_counter() >= deadline:
            break
        time.sleep(0.1)
    powers = np.asarray([row["power_w"] for row in samples])
    return {
        "label": "C1-W_AWQ_model_resident",
        "seconds": seconds,
        "samples": len(samples),
        "power_median_w": float(np.median(powers)),
        "power_mean_w": float(powers.mean()),
        "power_sd_w": float(powers.std(ddof=1)),
    }


def atomic_power_trace(block, prefill_samples, decode_samples):
    destination = POWER_TRACE_DIR / f"C1-W_ctx4096_block{block}.json"
    atomic_write_json(destination, {
        "fingerprint": EXPERIMENT_FINGERPRINT,
        "schema_version": SCHEMA_VERSION,
        "variant": "C1-W",
        "context": CONTEXT_TOKENS,
        "block": block,
        "prefill": prefill_samples,
        "decode": decode_samples,
    })
    return str(destination)


def run_c1w_energy(model, block, idle, warmup=False):
    variant = "C1-W"
    prompt = prompts["d50-s00"]
    enable_wrapper(False)
    cache = DynamicCache()
    ids, actual_hash = prompt_tensor(prompt, model.device)
    prefill_monitor = None
    decode_monitor = None
    try:
        clear_gpu()
        before = telemetry_snapshot()
        torch.cuda.reset_peak_memory_stats()
        prefill_monitor = GPUMonitor(0.1)
        prefill_monitor.start()
        start = time.perf_counter()
        with torch.inference_mode():
            model(
                input_ids=ids, past_key_values=cache, use_cache=True, logits_to_keep=1,
            )
        torch.cuda.synchronize()
        prefill_s = time.perf_counter() - start
        prefill_power = prefill_monitor.stop()
        prefill_lengths = cache_lengths(cache)
        post_prefill_storage = cache_storage_metrics(cache, "post_prefill")

        decode_monitor = GPUMonitor(0.1)
        decode_monitor.start()
        start = time.perf_counter()
        with torch.inference_mode():
            for step, token_id in enumerate(TRACE_IDS):
                token = torch.tensor([[token_id]], dtype=torch.long, device=model.device)
                absolute_position = CONTEXT_TOKENS + step
                position = torch.tensor([[absolute_position]], device=model.device)
                cache_position = torch.tensor([absolute_position], device=model.device)
                model(
                    input_ids=token,
                    position_ids=position,
                    cache_position=cache_position,
                    past_key_values=cache,
                    use_cache=True,
                    logits_to_keep=1,
                )
        torch.cuda.synchronize()
        decode_s = time.perf_counter() - start
        decode_power = decode_monitor.stop()
        update_log = list(CACHE_UPDATE_LOG)
        structure = validate_cache_path(
            cache, variant, prefill_lengths, FIXED_DECODE_TOKENS, update_log,
        )
        assert cache_lengths(cache) == [CONTEXT_TOKENS + FIXED_DECODE_TOKENS] * NUM_LAYERS
        assert_no_compression_state(cache)
        post_decode_storage = cache_storage_metrics(cache, "post_decode")
        after = telemetry_snapshot()
        total_energy = prefill_power["energy_j"] + decode_power["energy_j"]
        monitored_s = prefill_power["duration_s"] + decode_power["duration_s"]
        trace_file = "" if warmup else atomic_power_trace(
            block, prefill_power["samples"], decode_power["samples"],
        )
        return {
            "fingerprint": EXPERIMENT_FINGERPRINT,
            "schema_version": SCHEMA_VERSION,
            "session_id": SESSION_ID,
            "variant": variant,
            "context": CONTEXT_TOKENS,
            "block": block,
            "warmup": warmup,
            "input_ids_sha256": actual_hash,
            "trace_ids_sha256": TRACE_IDS_SHA256,
            "fixed_decode_tokens": FIXED_DECODE_TOKENS,
            "prefill_s": prefill_s,
            "decode_s": decode_s,
            "prefill_energy_j": prefill_power["energy_j"],
            "decode_energy_j": decode_power["energy_j"],
            "total_energy_j": total_energy,
            "idle_adjusted_energy_j": total_energy - idle["power_median_w"] * monitored_s,
            "decode_tok_s": FIXED_DECODE_TOKENS / decode_s,
            "avg_power_w": total_energy / monitored_s,
            "max_power_w": max(prefill_power["max_power_w"], decode_power["max_power_w"]),
            "power_samples": prefill_power["n_samples"] + decode_power["n_samples"],
            "idle_power_w": idle["power_median_w"],
            "temperature_before_c": before["temperature_c"],
            "temperature_after_c": after["temperature_c"],
            "sm_clock_before_mhz": before["sm_clock_mhz"],
            "sm_clock_after_mhz": after["sm_clock_mhz"],
            "memory_clock_before_mhz": before["memory_clock_mhz"],
            "memory_clock_after_mhz": after["memory_clock_mhz"],
            "power_limit_w": after["power_limit_w"],
            "throttle_before": before["throttle_reasons"],
            "throttle_after": after["throttle_reasons"],
            "trace_file": trace_file,
            **post_prefill_storage,
            **post_decode_storage,
            **structure,
        }
    finally:
        for monitor in (prefill_monitor, decode_monitor):
            if monitor is not None and monitor.running:
                try:
                    monitor.stop()
                except Exception:
                    monitor.running = False
        disable_wrapper()
        del cache, ids
        clear_gpu()


require_validation_gate()
energy_rows, energy_done = refresh_csv(
    ENERGY_ROW_DIR, ENERGY_CSV, ("variant", "context", "block"), ("C1-W",),
)
model = load_awq()
try:
    idle = measure_idle_telemetry()
    atomic_write_json(SAVE_DIR / f"c1w_idle_{SESSION_ID}.json", idle)
    run_c1w_energy(model, block=-1, idle=idle, warmup=True)
    for block in range(MEASURED_BLOCKS):
        key = ("C1-W", CONTEXT_TOKENS, block)
        if key in energy_done:
            continue
        row = run_c1w_energy(model, block=block, idle=idle, warmup=False)
        commit_row(
            ENERGY_ROW_DIR,
            ENERGY_CSV,
            row,
            ("variant", "context", "block"),
            ("C1-W",),
        )
        energy_done.add(key)
        print("energy", key, round(row["total_energy_j"], 2), "J")
finally:
    disable_wrapper()
    release_model()

energy_rows, _ = read_valid_shards(
    ENERGY_ROW_DIR, ("variant", "context", "block"), ("C1-W",),
)
assert len(energy_rows) == MEASURED_BLOCKS
assert {row["block"] for row in energy_rows} == set(range(MEASURED_BLOCKS))
for row in energy_rows:
    trace = json.loads(Path(row["trace_file"]).read_text(encoding="utf-8"))
    energies = []
    for phase in ("prefill", "decode"):
        times = np.asarray([sample["elapsed_s"] for sample in trace[phase]], dtype=float)
        powers = np.asarray([sample["power_w"] for sample in trace[phase]], dtype=float)
        assert len(times) >= 2 and np.all(np.diff(times) >= 0)
        if len(times) > 2:
            assert np.median(np.diff(times)) <= 0.25
        energies.append(float(np.trapezoid(powers, times)))
    assert np.isclose(sum(energies), row["total_energy_j"], rtol=1e-9, atol=1e-6)
print("Five-block C1-W energy control complete and traces reintegrated")

## 9. Quality Runner and Teacher-Forced Target Diagnostics

Generation is deterministic greedy decoding. A separate fresh-cache pass records first-target-token rank/NLL from prefill and teacher-forced continuation NLL after the first answer token, where the stored cache is active. Both passes validate every layer and actual prompt hash.

In [ ]:
def teacher_forced_target_metrics(model, variant, prompt):
    path = VARIANT_PATHS[variant]
    if path["wrapper"]:
        enable_wrapper(path["eviction"])
    else:
        assert LlamaAttention.forward is ORIGINAL_LLAMA_FORWARD
        PKV_LOG.clear()
        CACHE_UPDATE_LOG.clear()
    cache = new_cache(variant)
    ids, actual_hash = prompt_tensor(prompt, model.device)
    answer_ids = tokenizer.encode(prompt["target_code"], add_special_tokens=False)
    assert tokenizer.decode(answer_ids, skip_special_tokens=True) == prompt["target_code"]
    assert len(answer_ids) >= 2
    continuation_log_probabilities = []
    try:
        with torch.inference_mode():
            output = model(
                input_ids=ids, past_key_values=cache, use_cache=True, logits_to_keep=1,
            )
            prefill_lengths = cache_lengths(cache)
            first_logits = output.logits[0, -1].float()
            first_target_id = answer_ids[0]
            first_log_probability = float(
                F.log_softmax(first_logits, dim=-1)[first_target_id]
            )
            first_target_rank = int(
                (first_logits > first_logits[first_target_id]).sum().item() + 1
            )
            for index, current_token_id in enumerate(answer_ids[:-1]):
                token = torch.tensor(
                    [[current_token_id]], dtype=torch.long, device=model.device,
                )
                absolute_position = CONTEXT_TOKENS + index
                position = torch.tensor([[absolute_position]], device=model.device)
                cache_position = torch.tensor([absolute_position], device=model.device)
                output = model(
                    input_ids=token,
                    position_ids=position,
                    cache_position=cache_position,
                    past_key_values=cache,
                    use_cache=True,
                    logits_to_keep=1,
                )
                target_id = answer_ids[index + 1]
                log_probability = F.log_softmax(
                    output.logits[0, -1].float(), dim=-1,
                )[target_id]
                continuation_log_probabilities.append(float(log_probability))
        update_log = list(CACHE_UPDATE_LOG)
        structure = validate_cache_path(
            cache, variant, prefill_lengths, len(answer_ids) - 1, update_log,
        )
        assert len(continuation_log_probabilities) == len(answer_ids) - 1
        return {
            "input_ids_sha256": actual_hash,
            "answer_token_ids": answer_ids,
            "answer_tokens": len(answer_ids),
            "answer_first_token_logprob": first_log_probability,
            "answer_first_token_nll": -first_log_probability,
            "answer_first_token_rank": first_target_rank,
            "answer_first_token_top1": first_target_rank == 1,
            "scored_answer_tokens_after_first": len(continuation_log_probabilities),
            "answer_logprob_after_first_sum": float(sum(continuation_log_probabilities)),
            "answer_logprob_after_first_mean": float(np.mean(continuation_log_probabilities)),
            "answer_nll_after_first_mean": float(-np.mean(continuation_log_probabilities)),
            "teacher_forced_structural_pass": structure["structural_pass"],
            "teacher_forced_active_cache_updates": structure["active_decode_cache_updates"],
        }
    finally:
        if path["wrapper"]:
            disable_wrapper()
        del cache, ids
        clear_gpu()


def run_quality_case(model, variant, prompt):
    generation = run_greedy_path(model, variant, prompt)
    scoring = score_response(generation["response"], prompt["target_code"])
    target_metrics = teacher_forced_target_metrics(model, variant, prompt)
    assert generation["input_ids_sha256"] == target_metrics["input_ids_sha256"]
    assert generation["input_ids_sha256"] == prompt["input_ids_sha256"]
    return {
        "fingerprint": EXPERIMENT_FINGERPRINT,
        "schema_version": SCHEMA_VERSION,
        "session_id": SESSION_ID,
        "variant": variant,
        "scenario_id": prompt["scenario_id"],
        "depth": prompt["depth"],
        "context": CONTEXT_TOKENS,
        "record_count": prompt["record_count"],
        "query_window_tokens": prompt["query_window_tokens"],
        "target_key": prompt["target_key"],
        "target_code": prompt["target_code"],
        "input_ids_sha256": generation["input_ids_sha256"],
        "response": generation["response"],
        "output_tokens": len(generation["output_token_ids"]),
        "output_token_ids": generation["output_token_ids"],
        "extracted_codes": scoring["extracted_codes"],
        "unique_extracted_codes": scoring["unique_extracted_codes"],
        "retrieval_correct": scoring["retrieval_correct"],
        "format_correct": scoring["format_correct"],
        "legacy_target_anywhere": scoring["legacy_target_anywhere"],
        "failure_category": scoring["failure_category"],
        "structural_pass": generation["structure"]["structural_pass"],
        **{
            key: value for key, value in generation["structure"].items()
            if key != "structural_pass"
        },
        **generation["storage"],
        **target_metrics,
    }


print("Quality and target-diagnostic runners ready")

## 10. FP16 Model-Resident Phase: 60 C0 Cases

C0 receives the exact frozen AWQ-tokenizer input IDs. No historical easy-NIAH row is read or substituted. Each row is committed immediately.

In [ ]:
require_validation_gate()
quality_rows, quality_done = refresh_csv(
    QUALITY_ROW_DIR,
    QUALITY_CSV,
    ("variant", "scenario_id"),
    QUALITY_VARIANTS,
    require_prompt=True,
)
model = load_fp16()
try:
    for scenario_id in expected_scenario_ids:
        key = ("C0", scenario_id)
        if key in quality_done:
            continue
        row = run_quality_case(model, "C0", prompts[scenario_id])
        commit_row(
            QUALITY_ROW_DIR,
            QUALITY_CSV,
            row,
            ("variant", "scenario_id"),
            QUALITY_VARIANTS,
            require_prompt=True,
        )
        quality_done.add(key)
        print("quality", key, row["failure_category"], repr(row["response"]))
finally:
    disable_wrapper()
    release_model()
assert sum(key[0] == "C0" for key in quality_done) == 60
print("FP16 phase complete: 60/60 C0 cases")

## 11. AWQ Model-Resident Phase: 240 Matched Cases

C1-W, C2, C3-Ref, and C4-Ref run on every frozen scenario. Within each scenario their order is deterministic and randomized. Resumption skips only fully validated atomic shards.

In [ ]:
require_validation_gate()
quality_rows, quality_done = refresh_csv(
    QUALITY_ROW_DIR,
    QUALITY_CSV,
    ("variant", "scenario_id"),
    QUALITY_VARIANTS,
    require_prompt=True,
)
model = load_awq()
try:
    for scenario_id in expected_scenario_ids:
        depth_code = int(prompts[scenario_id]["depth"] * 100)
        scenario_index = int(scenario_id[-2:])
        order = np.random.default_rng(
            SEED + depth_code * 100 + scenario_index,
        ).permutation(AWQ_QUALITY_VARIANTS)
        for variant in order:
            variant = str(variant)
            key = (variant, scenario_id)
            if key in quality_done:
                continue
            row = run_quality_case(model, variant, prompts[scenario_id])
            commit_row(
                QUALITY_ROW_DIR,
                QUALITY_CSV,
                row,
                ("variant", "scenario_id"),
                QUALITY_VARIANTS,
                require_prompt=True,
            )
            quality_done.add(key)
            print("quality", key, row["failure_category"], repr(row["response"]))
finally:
    disable_wrapper()
    release_model()
for variant in AWQ_QUALITY_VARIANTS:
    assert sum(key[0] == variant for key in quality_done) == 60
print("AWQ phase complete: 240/240 matched cases")

## 12. Final Integrity, Bounded Uncertainty, and Paired Comparisons

This cell requires exactly 300 unique quality rows, 60 per configuration and 20 per depth. It reports bounded Wilson 95% intervals, paired prediction flips, Bonferroni-adjusted exact paired discordance tests, and deterministic depth-stratified paired bootstrap intervals for C2/C3-Ref/C4-Ref versus C1-W. Binary quality uncertainty is never represented by sample SD.

In [ ]:
def read_bool(series):
    if series.dtype == bool:
        return series
    normalized = series.astype(str).str.casefold()
    assert normalized.isin(["true", "false"]).all()
    return normalized == "true"


def wilson_interval(successes, n, confidence=0.95):
    assert 0 <= successes <= n and n > 0
    z = float(stats.norm.ppf(0.5 + confidence / 2))
    proportion = successes / n
    denominator = 1 + z * z / n
    center = (proportion + z * z / (2 * n)) / denominator
    half = z * math.sqrt(
        proportion * (1 - proportion) / n + z * z / (4 * n * n)
    ) / denominator
    return max(0.0, center - half), min(1.0, center + half)


def stratified_paired_bootstrap(pair, repetitions=10_000):
    rng = np.random.default_rng(SEED + 90_000)
    estimates = np.empty(repetitions, dtype=float)
    strata = [group.reset_index(drop=True) for _, group in pair.groupby("depth", sort=True)]
    assert [len(group) for group in strata] == [20, 20, 20]
    for repetition in range(repetitions):
        differences = []
        for group in strata:
            indices = rng.integers(0, len(group), len(group))
            sampled = group.iloc[indices]
            differences.extend((sampled["test"] - sampled["baseline"]).tolist())
        estimates[repetition] = np.mean(differences)
    estimate = float((pair["test"] - pair["baseline"]).mean())
    low, high = np.quantile(estimates, [0.025, 0.975])
    return estimate, float(np.clip(low, -1, 1)), float(np.clip(high, -1, 1))


require_validation_gate()
rows, keys = read_valid_shards(
    QUALITY_ROW_DIR,
    ("variant", "scenario_id"),
    QUALITY_VARIANTS,
    require_prompt=True,
)
assert len(rows) == 300 and len(keys) == 300
quality = pd.DataFrame(rows)
assert not quality.duplicated(["variant", "scenario_id"]).any()
assert set(quality.variant) == set(QUALITY_VARIANTS)
assert quality.groupby("variant").size().eq(60).all()
assert quality.groupby(["variant", "depth"]).size().eq(20).all()
assert quality.groupby("scenario_id").variant.nunique().eq(5).all()
assert quality.groupby("scenario_id").input_ids_sha256.nunique().eq(1).all()
assert quality.groupby("scenario_id").target_code.nunique().eq(1).all()
assert read_bool(quality.structural_pass).all()
assert read_bool(quality.teacher_forced_structural_pass).all()
assert set(quality.failure_category) <= {
    "pass", "multiple_code", "wrong_code", "truncated_code", "no_code",
}
for column in ("retrieval_correct", "format_correct", "legacy_target_anywhere"):
    quality[column] = read_bool(quality[column])

interval_rows = []
for metric in ("retrieval_correct", "format_correct", "legacy_target_anywhere"):
    for grouping in (("variant",), ("variant", "depth")):
        for key, group in quality.groupby(list(grouping), sort=True):
            if not isinstance(key, tuple):
                key = (key,)
            successes = int(group[metric].sum())
            low, high = wilson_interval(successes, len(group))
            row = dict(zip(grouping, key))
            row.update({
                "metric": metric,
                "successes": successes,
                "n": len(group),
                "rate": successes / len(group),
                "wilson95_low": low,
                "wilson95_high": high,
            })
            interval_rows.append(row)
intervals = pd.DataFrame(interval_rows)
intervals.to_csv(ANALYSIS_DIR / "quality_wilson_intervals.csv", index=False)

flip_rows = []
comparison_rows = []
baseline = quality[quality.variant == "C1-W"][
    ["scenario_id", "depth", "input_ids_sha256", "retrieval_correct"]
].rename(columns={"retrieval_correct": "baseline"})
for variant in ("C2", "C3-Ref", "C4-Ref"):
    test = quality[quality.variant == variant][
        ["scenario_id", "depth", "input_ids_sha256", "retrieval_correct"]
    ].rename(columns={
        "retrieval_correct": "test",
        "depth": "test_depth",
        "input_ids_sha256": "test_hash",
    })
    pair = baseline.merge(test, on="scenario_id", validate="one_to_one")
    assert len(pair) == 60
    assert (pair.depth == pair.test_depth).all()
    assert (pair.input_ids_sha256 == pair.test_hash).all()
    pair["baseline"] = pair.baseline.astype(int)
    pair["test"] = pair.test.astype(int)
    for depth_label, group in [("all", pair), *list(pair.groupby("depth", sort=True))]:
        n00 = int(((group.baseline == 0) & (group.test == 0)).sum())
        n01 = int(((group.baseline == 0) & (group.test == 1)).sum())
        n10 = int(((group.baseline == 1) & (group.test == 0)).sum())
        n11 = int(((group.baseline == 1) & (group.test == 1)).sum())
        discordant = n01 + n10
        p_exact = (
            float(stats.binomtest(min(n01, n10), discordant, 0.5).pvalue)
            if discordant else 1.0
        )
        flip_rows.append({
            "comparison": f"{variant}-C1-W",
            "depth": depth_label,
            "n": len(group),
            "both_wrong_n00": n00,
            "gain_n01": n01,
            "loss_n10": n10,
            "both_correct_n11": n11,
            "paired_difference": (n01 - n10) / len(group),
            "mcnemar_exact_two_sided_p": p_exact,
        })
    estimate, low, high = stratified_paired_bootstrap(pair)
    comparison_rows.append({
        "comparison": f"{variant}-C1-W",
        "n": 60,
        "method": "depth-stratified paired bootstrap, 10000 repetitions",
        "paired_retrieval_difference": estimate,
        "ci95_low": low,
        "ci95_high": high,
    })
flips = pd.DataFrame(flip_rows)
flips["mcnemar_p_bonferroni_12"] = np.minimum(
    1.0, flips["mcnemar_exact_two_sided_p"] * len(flips),
)
comparisons = pd.DataFrame(comparison_rows)
flips.to_csv(ANALYSIS_DIR / "paired_prediction_flips.csv", index=False)
comparisons.to_csv(ANALYSIS_DIR / "paired_depth_stratified_comparisons.csv", index=False)

failure_table = quality.groupby(
    ["variant", "depth", "failure_category"], sort=True,
).size().rename("count").reset_index()
failure_table.to_csv(ANALYSIS_DIR / "failure_categories.csv", index=False)
diagnostic_summary = quality.groupby(["variant", "depth"], sort=True).agg(
    output_tokens_mean=("output_tokens", "mean"),
    first_target_rank_median=("answer_first_token_rank", "median"),
    first_target_nll_mean=("answer_first_token_nll", "mean"),
    continuation_nll_mean=("answer_nll_after_first_mean", "mean"),
    format_rate=("format_correct", "mean"),
    legacy_target_anywhere_rate=("legacy_target_anywhere", "mean"),
).reset_index()
diagnostic_summary.to_csv(ANALYSIS_DIR / "quality_diagnostics.csv", index=False)

energy_rows, energy_keys = read_valid_shards(
    ENERGY_ROW_DIR,
    ("variant", "context", "block"),
    ("C1-W",),
)
assert len(energy_rows) == MEASURED_BLOCKS and len(energy_keys) == MEASURED_BLOCKS
assert energy_keys == {
    ("C1-W", CONTEXT_TOKENS, block) for block in range(MEASURED_BLOCKS)
}
for row in energy_rows:
    trace_path = Path(row["trace_file"])
    assert trace_path.exists(), trace_path
    assert trace_path.resolve().parent == POWER_TRACE_DIR.resolve()
    trace = json.loads(trace_path.read_text(encoding="utf-8"))
    assert trace["fingerprint"] == EXPERIMENT_FINGERPRINT
    assert trace["schema_version"] == SCHEMA_VERSION
    assert trace["variant"] == "C1-W"
    assert trace["context"] == CONTEXT_TOKENS
    assert trace["block"] == row["block"]
    reintegrated = 0.0
    for phase in ("prefill", "decode"):
        times = np.asarray([sample["elapsed_s"] for sample in trace[phase]], dtype=float)
        powers = np.asarray([sample["power_w"] for sample in trace[phase]], dtype=float)
        assert len(times) >= 2 and np.all(np.diff(times) >= 0)
        if len(times) > 2:
            assert np.median(np.diff(times)) <= 0.25
        reintegrated += float(np.trapezoid(powers, times))
    assert np.isclose(reintegrated, row["total_energy_j"], rtol=1e-9, atol=1e-6)

refresh_csv(
    QUALITY_ROW_DIR,
    QUALITY_CSV,
    ("variant", "scenario_id"),
    QUALITY_VARIANTS,
    require_prompt=True,
)
final_record = {
    "passed": True,
    "fingerprint": EXPERIMENT_FINGERPRINT,
    "schema_version": SCHEMA_VERSION,
    "quality_rows": 300,
    "energy_rows": len(energy_rows),
    "unique_quality_keys": 300,
    "paired_prompt_hash_gate": True,
    "energy_trace_reintegration_gate": True,
    "bounded_quality_uncertainty": "Wilson 95% intervals",
    "paired_comparison_method": "depth-stratified paired bootstrap",
    "completed_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
}
atomic_write_json(ANALYSIS_DIR / "final_integrity.json", final_record)
display(intervals)
display(flips)
display(comparisons)
print(json.dumps(final_record, indent=2))

## Completion Gate

Accept this run only when `analysis/final_integrity.json` reports 300 quality rows, five energy rows, 300 unique keys, and a passed paired-prompt-hash gate. Keep all V1-V4 pilot directories and `CSE407_llama_v5_final/` immutable. Do not merge pilot results into this analysis.